In [1]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path

# Path set karo
project_root = Path(r"D:\projects\RAG Agent")
sys.path.insert(0, str(project_root / "src"))

# .env file load karo
load_dotenv(dotenv_path=project_root / ".env")

# Check karo ki key aayi ya nahi
key = os.environ.get("RAG_API_KEY")
if key:
    print(f"✅ Key loaded: {key[:6]}...")
else:
    print("❌ Key still missing! Check your .env file.")

✅ Key loaded: AQ.Ab8...


In [2]:
import sys
print(sys.executable)

D:\anaconda\programs\envs\rag-agent\python.exe


In [3]:
!{sys.executable} -m pip install langchain langchain-community langchain-text-splitters faiss-cpu pypdf numpy python-dotenv langchain-google-genai

In [4]:
import recipe1_ingest
import recipe2_embed
import recipe3_build_index
import recipe4_retrieve

print("1. INGESTION")
recipe1_ingest.main()

print("\n2. EMBEDDING (API Call - wait 10-20s)")
recipe2_embed.main()

print("\n3. BUILD INDEX")
recipe3_build_index.main()

print("\n4. RETRIEVE")
results = recipe4_retrieve.retrieve("What is this PDF about?", top_k=3)

for r in results:
    print(f"\nScore: {r['score']} | Page: {r['page']}")
    print(f"Text: {r['text'][:200]}...")

if results and results[0]['score'] > 0.3:
    print("\n✅ PHASE-1 MILESTONE: PASS")
else:
    print("\n❌ PHASE-1 MILESTONE: FAIL (Score too low)")

1. INGESTION
[ingest] test.pdf: 14 pages, 8311 chars extracted
[ingest] test.pdf: 18 chunks
[ingest] OK: 18 chunks -> D:\projects\RAG Agent\data\chunks\chunks.jsonl

2. EMBEDDING (API Call - wait 10-20s)


usage: ipykernel_launcher.py [-h] [--force] [--sleep SLEEP]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\pmthe\AppData\Roaming\jupyter\runtime\kernel-39359950-645e-489c-9591-6e63f3366288.json


SystemExit: 2

D:\anaconda\programs\envs\rag-agent\Lib\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
import sys
import recipe2_embed
import recipe3_build_index
import recipe4_retrieve

# Jupyter ke default arguments ko clear karo
sys.argv = ['recipe2_embed.py']

print("2. EMBEDDING (API Call - wait 10-20s)")
recipe2_embed.main()

print("\n3. BUILD INDEX")
recipe3_build_index.main()

print("\n4. RETRIEVE")
results = recipe4_retrieve.retrieve("What is this PDF about?", top_k=3)

for r in results:
    print(f"\nScore: {r['score']} | Page: {r['page']}")
    print(f"Text: {r['text'][:200]}...")

if results and results[0]['score'] > 0.3:
    print("\n✅ PHASE-1 MILESTONE: PASS")
else:
    print("\n❌ PHASE-1 MILESTONE: FAIL (Score too low)")

2. EMBEDDING (API Call - wait 10-20s)
[embed] corpus unchanged (hash 60c30486e8...), skipping. Use --force to override.

3. BUILD INDEX
[index] OK: 18 vectors -> D:\projects\RAG Agent\index\faiss.index

4. RETRIEVE

Score: 0.624 | Page: 14
Text: McSwane, D., Rue, N. R., & Linton, R. (2003). Food Safety and Sanitation. New Jersey: Prentice 
Hall
Ninemeier, J. D., & Hayes, D. K. (2006). Restaurant Operations Management Principles and 
Practice....

Score: 0.5997 | Page: 4
Text: attract clientele that are cost-conscious, though not necessarily budget
travelers. These hotels are also preferred by people who do not want to stay
in the heart of city as they want to be away from ...

Score: 0.5949 | Page: 2
Text: Types of Accommodation
1. Location:-
 City-centre/Downtown hotels
 Suburban hotels
 Motels
 Airport hotels
 Resort hotels
 Forest hotels
 Floatels
1. Size of property
 Small hotels
 Medium si...

✅ PHASE-1 MILESTONE: PASS


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# 1. Setup the LLM (Gemini 2.5 Flash)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ.get("RAG_API_KEY"),
    temperature=0.2 # Kam temperature matlab kam bakwaas, zyada factual answer
)

# 2. User's query
query = "What is the difference between a motel and a resort hotel?"

# 3. Retrieval (Step 1 of RAG)
print("Retrieving context...")
results = recipe4_retrieve.retrieve(query, top_k=3)

# 4. Context converting to string
context_text = "\n\n".join([f"[Page {r['page']}] {r['text']}" for r in results])

# 5. The Prompt (Instruction for the LLM)
prompt = f"""
You are a helpful assistant. Answer the user's question using ONLY the provided context.
If the answer is not in the context, say "I don't know based on the provided text."

Context:
{context_text}

Question: {query}

Answer:
"""

# 6. Generation (Step 2 of RAG)
print("Generating answer...\n")
response = llm.invoke([HumanMessage(content=prompt)])

print("=" * 50)
print("FINAL ANSWER:")
print("=" * 50)
print(response.content)

Retrieving context...
Generating answer...

FINAL ANSWER:
Motels are located alongside the highways and road junctions, catering to the hospitality requirements of road travellers for overnight stays. They are normally economical and provide comfortable bedrooms, along with facilities like a motor garage, filling station, and parking area for motor vehicles.

Resorts, on the other hand, are located in natural and man-made sites and are places used for relaxation or recreation, attracting visitors for holidays or vacations.


---
# Phase 4 - `ingest_cli.py` + source-filtered retrieval

Run the cells below **top to bottom**. Costs:

| Cell | What it does | Gemini calls |
|------|--------------|--------------|
| 4.1 | helpers + reload of the Phase-4 modules | 0 |
| 4.2 | offline self-tests | 0 |
| 4.3 | `ingest_cli list` | 0 |
| 4.4 | invalid inputs must be rejected (exit code 1) | 0 |
| 4.5 | re-add an already indexed PDF (idempotent) | 0 if the corpus is unchanged |
| 4.6 | add a NEW pdf (set `NEW_PDF`) | **re-embeds the whole corpus** (recipe2 has no per-chunk cache) |
| 4.7 | list again | 0 |
| 4.8 | filtered vs unfiltered retrieval | 2 (query embeddings) |
| 4.9 | unknown source must return "not found" | 0 |
| 4.10 | filtered grounded QA | 1 query embedding + 1 LLM call |
| 4.11 | `ingest_cli sync` | 0 if in sync |

Prerequisite: the Phase-4 files (`recipe1_ingest.py`, `recipe4_retrieve.py`, `qa_chain.py`, `tools.py`, `ingest_cli.py`) are in `D:\projects\RAG Agent\src`.

### 4.1 Setup: CLI helper + reload

In [7]:
import importlib
import os
import subprocess
import sys
import tempfile
from pathlib import Path

# same project root as the first cell (re-asserted so this section can run on its own)
project_root = Path(r"D:\projects\RAG Agent")
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

INGEST_CLI = project_root / "src" / "ingest_cli.py"


def run_cli(*args):
    """Run ingest_cli.py in a subprocess and stream its output into this cell. Returns the exit code.

    A subprocess (not ingest_cli.main()) is used on purpose: the CLI reads sys.argv with argparse,
    which breaks inside Jupyter, and it launches recipe1/2/3 as child processes whose output
    would otherwise go to the Jupyter server console instead of this notebook.
    """
    cmd = [sys.executable, "-u", str(INGEST_CLI), *[str(a) for a in args]]
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    print("$ ingest_cli.py", *[str(a) for a in args], "\n")
    proc = subprocess.Popen(
        cmd, cwd=str(project_root), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line, end="")
        rc = proc.wait()
    except KeyboardInterrupt:  # kernel interrupt: do not leave an orphan pipeline running
        proc.kill()
        raise
    print(f"\n[exit code {rc}]")
    return rc


def reload_project_modules():
    """Pick up the Phase-4 versions of the modules and drop stale caches after an ingest."""
    for name in ("config", "recipe4_retrieve", "qa_chain", "tools"):  # dependency order
        if name in sys.modules:
            importlib.reload(sys.modules[name])
        else:
            importlib.import_module(name)
    sys.modules["qa_chain"]._chunk_id_lookup.cache_clear()  # chunk_id cache would miss new chunks


try:
    reload_project_modules()
except ImportError as e:
    raise SystemExit(f"Phase-4 files are missing or old in {project_root / 'src'}: {e}")

import config
import recipe4_retrieve
import qa_chain
import tools

assert hasattr(recipe4_retrieve, "clean_filter"), "recipe4_retrieve.py is still the Phase-1 version"
assert INGEST_CLI.exists(), f"missing {INGEST_CLI}"
print("Phase-4 modules loaded. Index files folder:", config.INDEX_DIR)

Phase-4 modules loaded. Index files folder: D:\projects\RAG Agent\index


### 4.2 Offline self-tests (no API, no index needed)

In [8]:
assert recipe4_retrieve._selftest() == 0
assert tools._self_test() == 0

recipe4 selftest: OK
PASS {'amount': 42.5, 'location': 'Rourkela Station', 'date': '2026-10-30'} -> SUCCESS: Expense of $42.50 filed for Rourkela Station on 2026-10-30.
PASS {'amount': -5, 'location': 'X', 'date': '2026-10-30'} -> ERROR: Invalid input. amount must be greater than 0, got -5.0.
PASS {'amount': 0, 'location': 'X', 'date': '2026-10-30'} -> ERROR: Invalid input. amount must be greater than 0, got 0.0.
PASS {'amount': 10, 'location': '  ', 'date': '2026-10-30'} -> ERROR: Invalid input. location must be a non-empty string.
PASS {'amount': 10, 'location': 'X', 'date': '30-10-2026'} -> ERROR: Invalid input. date must be YYYY-MM-DD, got '30-10-2026'.
PASS {'amount': 10, 'location': 'X', 'date': '2026-02-30'} -> ERROR: Invalid input. 2026-02-30 is not a real calendar date (use YYYY-MM-DD).
PASS {'amount': 5000000, 'location': 'X', 'date': '2026-10-30'} -> ERROR: Invalid input. amount 5000000.00 exceeds the 1,000,000 limit; confirm the amount with the user.
PASS tool 'search_docum

### 4.3 `list`: what is indexed right now?

In [9]:
assert run_cli("list") == 0

$ ingest_cli.py list 

Indexed sources (index\chunks_meta.json):
  test.pdf     18 chunks

[exit code 0]


### 4.4 Invalid inputs must be rejected
Each case must exit with code **1** and must not touch `data\pdfs`. No API calls are made.

In [10]:
pdf_dir = config.PDF_DIR
before = sorted(p.name for p in pdf_dir.iterdir())

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    not_pdf = tmp / "notes.txt"
    not_pdf.write_text("hello", encoding="utf-8")
    fake_pdf = tmp / "fake.pdf"
    fake_pdf.write_bytes(b"this is not a pdf")
    empty_pdf = tmp / "empty.pdf"
    empty_pdf.write_bytes(b"")

    cases = {
        "missing file": tmp / "does_not_exist.pdf",
        "wrong extension": not_pdf,
        "fake pdf (no %PDF- header)": fake_pdf,
        "empty file": empty_pdf,
    }
    for label, path in cases.items():
        print("=" * 60, "\nCASE:", label)
        assert run_cli("add", path) == 1, f"'{label}' should have been rejected"

after = sorted(p.name for p in pdf_dir.iterdir())
assert before == after, "a rejected add must not change data\\pdfs"
print("\nAll invalid inputs were rejected and the corpus is unchanged.")

CASE: missing file
$ ingest_cli.py add C:\Users\pmthe\AppData\Local\Temp\tmpyobaqk1n\does_not_exist.pdf 

ERROR: file not found: C:\Users\pmthe\AppData\Local\Temp\tmpyobaqk1n\does_not_exist.pdf

[exit code 1]
CASE: wrong extension
$ ingest_cli.py add C:\Users\pmthe\AppData\Local\Temp\tmpyobaqk1n\notes.txt 

ERROR: only .pdf files are accepted

[exit code 1]
CASE: fake pdf (no %PDF- header)
$ ingest_cli.py add C:\Users\pmthe\AppData\Local\Temp\tmpyobaqk1n\fake.pdf 

ERROR: not a real PDF (missing %PDF- header)

[exit code 1]
CASE: empty file
$ ingest_cli.py add C:\Users\pmthe\AppData\Local\Temp\tmpyobaqk1n\empty.pdf 

ERROR: file is empty

[exit code 1]

All invalid inputs were rejected and the corpus is unchanged.


### 4.5 Re-add an already indexed PDF (idempotent)
Same content, same name: nothing is copied, recipe2's hash guard skips embedding if the corpus is unchanged, the index is rebuilt locally and verified.

In [11]:
existing = sorted(config.PDF_DIR.glob("*.pdf"), key=lambda p: p.name.lower())
assert existing, f"no PDFs in {config.PDF_DIR}; run the Phase-1 cells first"
assert run_cli("add", existing[0]) == 0

$ ingest_cli.py add D:\projects\RAG Agent\data\pdfs\test.pdf 

[ingest] 'test.pdf' is already in the corpus (same content); skipping copy.
[ingest] corpus file: D:\projects\RAG Agent\data\pdfs\test.pdf

>>> recipe1_ingest.py
D:\projects\RAG Agent\src\recipe1_ingest.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
[ingest] test.pdf: 14 pages, 8311 chars extracted
[ingest] test.pdf: 18 chunks
[ingest] OK: 18 chunks -> D:\projects\RAG Agent\data\chunks\chunks.jsonl
[ingest] 'test.pdf': 18 chunks (corpus: 18 chunks in 1 sources)

>>> recipe1_ingest.py
D:\projects\RAG Agent\src\recipe1_ingest.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langcha

### 4.6 Add a NEW PDF
Set `NEW_PDF` to a real file, then run. This **re-embeds every chunk** of the corpus (Gemini calls scale with corpus size). Use `--overwrite` only to replace a *different* file that has the same name.

In [12]:
NEW_PDF = r""  # e.g. r"D:\downloads\policy.pdf"

if not NEW_PDF.strip():
    print("NEW_PDF is empty: skipping. Set it to a .pdf path and re-run this cell.")
elif not Path(NEW_PDF).exists():
    print(f"File not found: {NEW_PDF}")
else:
    rc = run_cli("add", NEW_PDF)          # add "--overwrite" here to replace a same-named file
    qa_chain._chunk_id_lookup.cache_clear()  # new chunks must be visible to citation enrichment
    assert rc == 0, "add failed; the output above says why (run 'sync' after fixing API/quota issues)"

NEW_PDF is empty: skipping. Set it to a .pdf path and re-run this cell.


### 4.7 `list` again: the new file should show up

In [13]:
assert run_cli("list") == 0
sources = recipe4_retrieve.list_sources()
print("\nSources visible to retrieve():", sources)
assert sources, "index is empty"

$ ingest_cli.py list 

Indexed sources (index\chunks_meta.json):
  test.pdf     18 chunks

[exit code 0]

Sources visible to retrieve(): {'test.pdf': 18}


### 4.8 Filtered vs unfiltered retrieval
`SOURCE` empty = first indexed file. Every filtered hit must come from that file only.

In [14]:
SOURCE = ""                                   # e.g. "policy.pdf" (case and .pdf are optional)
QUERY = "What is this PDF about?"

sources = recipe4_retrieve.list_sources()
SOURCE = SOURCE.strip() or next(iter(sources))
print("Filtering on:", SOURCE, "\n")

res_all = recipe4_retrieve.retrieve(QUERY, top_k=3)
res_flt = recipe4_retrieve.retrieve(QUERY, top_k=3, filter_source=SOURCE)

print("UNFILTERED")
for r in res_all:
    print(f"  {r['score']:.3f} | {r['source']} p.{r['page']}")
print("FILTERED")
for r in res_flt:
    print(f"  {r['score']:.3f} | {r['source']} p.{r['page']} | {r['text'][:80]!r}")

assert res_flt, "filter returned nothing although the source is indexed"
want = recipe4_retrieve.clean_filter(SOURCE)
assert all(recipe4_retrieve.normalize_source(r["source"]) in (want, want + ".pdf") for r in res_flt), \
    "a chunk from another file leaked through the filter"
print("\nOK: the filtered results contain only", SOURCE)

Filtering on: test.pdf 

UNFILTERED
  0.624 | test.pdf p.14
  0.600 | test.pdf p.4
  0.595 | test.pdf p.2
FILTERED
  0.624 | test.pdf p.14 | 'McSwane, D., Rue, N. R., & Linton, R. (2003). Food Safety and Sanitation. New Je'
  0.600 | test.pdf p.4 | 'attract clientele that are cost-conscious, though not necessarily budget\ntravele'
  0.595 | test.pdf p.2 | 'Types of Accommodation\n1. Location:-\n\uf0d8 City-centre/Downtown hotels\n\uf0d8 Suburban ho'

OK: the filtered results contain only test.pdf


### 4.9 Unknown source: graceful "not found"
`retrieve()` returns `[]` and `answer_question()` explains which sources exist. No Gemini call is made.

In [15]:
assert recipe4_retrieve.retrieve("anything", top_k=3, filter_source="does_not_exist.pdf") == []

res = qa_chain.answer_question("anything", filter_source="does_not_exist.pdf")
print(res.answer)
print("warnings:", res.warnings)
assert res.used_context is False and res.answerable is False

# values that mean "no filter" must not be treated as a file name
assert recipe4_retrieve.clean_filter("None") is None and recipe4_retrieve.clean_filter("all") is None
print("\nOK")

No indexed document matches the source filter 'does_not_exist.pdf'. Indexed sources: test.pdf.
warnings: ["search restricted to source 'does_not_exist.pdf'"]

OK


### 4.10 Filtered grounded QA (1 LLM call)
Citations must only come from the filtered file.

In [16]:
QA_QUESTION = "What is the difference between a motel and a resort hotel?"   # change to a question about SOURCE

res = qa_chain.answer_question(QA_QUESTION, top_k=3, filter_source=SOURCE)

print("ANSWER:\n", res.answer, "\n")
print("sources      :", [f"{s.source} p.{s.page}" for s in res.sources])
print("used_context :", res.used_context, "| answerable:", res.answerable, "| confidence:", res.confidence)
print("top_score    :", res.top_score, "| threshold:", res.threshold)
print("warnings     :", res.warnings)

want = recipe4_retrieve.clean_filter(SOURCE)
assert all(recipe4_retrieve.normalize_source(s.source) in (want, want + ".pdf") for s in res.sources), \
    "citation from a file outside the filter"

ANSWER:
 Motels are located along highways and road junctions, are meant for overnight stays for road travelers, are typically economical, and have facilities such as parking, motor garages, and filling stations. In contrast, resorts are located in natural or man-made sites and are places used for relaxation or recreation that attract visitors for holidays or vacations. 

sources      : ['test.pdf p.5']
used_context : True | answerable: True | confidence: high
top_score    : 0.7186 | threshold: 0.3
warnings     : ["search restricted to source 'test.pdf'"]


### 4.11 `sync`
Use after a failed `add` (API key, quota, network) or when `list` warns that `chunks.jsonl` is stale. Embedding batches are checkpointed, so a re-run resumes instead of starting over; if nothing changed it makes no API calls.

In [17]:
assert run_cli("sync") == 0

$ ingest_cli.py sync 


>>> recipe1_ingest.py
D:\projects\RAG Agent\src\recipe1_ingest.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
[ingest] test.pdf: 14 pages, 8311 chars extracted
[ingest] test.pdf: 18 chunks
[ingest] OK: 18 chunks -> D:\projects\RAG Agent\data\chunks\chunks.jsonl
[ingest] corpus = 18 chunks. recipe2 has no per-chunk cache: if the corpus changed, ALL chunks are re-embedded (unchanged corpus = zero API calls).

>>> recipe2_embed.py
[embed] corpus unchanged (hash 60c30486e8...), skipping. Use --force to override.

>>> recipe3_build_index.py
[index] OK: 18 vectors -> D:\projects\RAG Agent\index\faiss.index

OK: index is in sync.
Indexed sources (index\chunks_meta.json):
  test.pdf     18 chunks

[exit code 0]
